# Ordered Logistic Regression Results for Adoption Predictors Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset described by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will allow us to inspect the available record sets, fields, and overall dataset metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Instantiate the Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset metadata summary
print(f"{metadata.name}: {metadata.description}")
print(f"Authors: {[author for author in (metadata.author or [])]}")
print(f"Coverage: {metadata.spatialCoverage}")
print(f"Temporal: {metadata.temporalCoverage}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Explore available record sets, fields, their `@id`s and basic structure. This step ensures we know what collections of data (record sets) and fields (columns/attributes) are present before extracting records.

*We will retrieve and display all record sets, then for each, display the fields and columns and their `@id`s.*

In [ ]:
# List all available record sets and their fields/columns by `@id`
print('Available record sets in the dataset:')
record_sets = []
for rset in dataset.record_sets:
    print(f"- Record Set name: {getattr(rset, 'name', 'N/A')}  |  @id: {rset.id}")
    record_sets.append(rset.id)
    if hasattr(rset, 'fields') and rset.fields:
        print('  Fields:')
        for field in rset.fields:
            print(f"    - {getattr(field, 'name', 'N/A')}  @id: {field.id}")
    if hasattr(rset, 'columns') and rset.columns:
        print('  Columns:')
        for column in rset.columns:
            print(f"    - {getattr(column, 'name', 'N/A')}  @id: {column.id}")
    print('')

# For quick demonstration, list some records for the first record set (if any)
if record_sets:
    first_rset_id = record_sets[0]
    print(f'Example records from record set: {first_rset_id}')
    try:
        for ix, rec in zip(range(5), dataset.records(record_set=first_rset_id)):
            print(rec)
    except Exception as e:
        print(f"Cannot fetch records for {first_rset_id}: {e}")

## 3. Data Extraction
We will load all records for each record set into a Pandas DataFrame.
Record sets and fields are referenced strictly by their `@id`s for consistency.

In [ ]:
dataframes = {}
# We'll limit to first 3 record sets for sample output if many exist
sampled_record_sets = record_sets[:3]
for record_set_id in sampled_record_sets:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in {record_set_id}:\n", df.columns.tolist())
    print(df.head(2))

# Pick the main record set for further exploration
record_set_id = sampled_record_sets[0] if sampled_record_sets else None
if record_set_id:
    df_main = dataframes[record_set_id]
    print(f"\nShape: {df_main.shape}, columns: {df_main.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Let us select a numeric field (by its `@id`), filter on this field, normalize its distribution, and, if possible, group and aggregate by a categorical field (also referenced by `@id`).
*If the record set has no numeric columns, this section will skip computations but display actual available columns.*

In [ ]:
# Inspect columns of our main DataFrame
if record_set_id:
    df = dataframes[record_set_id]
    print(f"Columns: {df.columns.tolist()}")

    # Try to select a numeric field and group field by inspecting the dataframe dtypes
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    group_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])]

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field by @id: {numeric_field_id}")

        # Use a threshold as in the template
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().sum() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a categorical/grouping field
        group_field_id = group_candidates[0] if group_candidates else None
        if group_field_id:
            print(f"Grouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
    else:
        print("No numeric fields found in this record set. Available columns are:")
        print(df.columns.tolist())

## 5. Visualization
Visualize the distribution of the selected numeric field and its values by group if grouping is feasible.
Plots are generated only if above EDA selected numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if record_set_id and 'numeric_field_id' in locals():
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of numeric field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group if available
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print('Visualization skipped: no numeric field found in selected record set.')

## 6. Conclusion
In this notebook, we demonstrated how to utilize the Croissant schema and `mlcroissant` to:

- Load dataset metadata for a FAIR^2 public dataset.
- Enumerate available record sets, fields, and their unique `@id`s.
- Load records from a record set into a DataFrame and perform basic EDA by referencing Croissant schema `@id`s throughout.
- Visualize numeric data distributions and group-based summaries.

This approach can be easily adapted for analysis of any dataset described using a Croissant schema, supporting strong reproducibility and semantic referencing of all dataset entities via their persistent identifiers.